# RoMa-only ΔE (dense flow, no SGM)

This notebook drops the essential-matrix / rectification / SGM pipeline entirely.
RoMa's **dense warp already is the per-pixel correspondence**, so we just resample B
into A's frame with that warp, keep the confident pixels, and compute ΔE2000 directly.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

from PIL import Image
from romatch import tiny_roma_v1_outdoor
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("/run/media/gabriele/discone/Files/Magistrale/TESI/Codice")
sys.path.insert(0, str(ROOT / "Tests"))

from thesis_lib_shared.io import read_mat_image
from thesis_lib_shared.metrics import rgb_to_lab, delta_e_cie2000, delta_e_summary

In [ ]:
PAIRED = ROOT / "data/iphone2samsung_or/pair-np-dv/paired"

img_a = read_mat_image(PAIRED / "iphone-x/dng/00.mat")
img_b = read_mat_image(PAIRED / "samsung-s9/dng/00.mat")
img_a_orig, img_b_orig = img_a, img_b
print("loaded A", img_a.shape, "B", img_b.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_a); axes[0].set_title(f"iPhone (A) {img_a.shape}"); axes[0].axis("off")
axes[1].imshow(img_b); axes[1].set_title(f"Samsung (B) {img_b.shape}"); axes[1].axis("off")
plt.tight_layout(); plt.show()

# RoMa dense match

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tiny_roma = tiny_roma_v1_outdoor(device=device)
tiny_roma.train(False)


def to_pil(img: np.ndarray) -> Image.Image:
    """RGB float [0,1] or uint8 array -> PIL.Image (uint8 RGB), as TinyRoMa expects."""
    arr = img.astype(np.float32)
    if arr.max() <= 1.0:
        arr = arr * 255.0
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="RGB")


def to01(img: np.ndarray) -> np.ndarray:
    """Return a float32 copy in [0, 1] regardless of input scale."""
    a = img.astype(np.float32)
    return a / 255.0 if a.max() > 1.0 else a

In [ ]:
# Aspect-preserving working frame (square pixels), then a single dense forward pass.
ROMA_LONG = 1024
h0, w0 = img_a_orig.shape[:2]
s_roma = ROMA_LONG / max(h0, w0)
ROMA_W, ROMA_H = round(w0 * s_roma), round(h0 * s_roma)

img_a = cv2.resize(img_a_orig, (ROMA_W, ROMA_H))
img_b = cv2.resize(img_b_orig, (ROMA_W, ROMA_H))
h_a, w_a = img_a.shape[:2]
print(f"working frame: {w_a}x{h_a} (scale={s_roma:.5f})")

# warp (h_a, w_a, 4): channels 0:2 = this A-pixel's own normalized coords,
#                     channels 2:4 = the matched location in B (normalized [-1, 1]).
# certainty (h_a, w_a): per-pixel confidence of the dense match.
warp, certainty = tiny_roma.match(to_pil(img_a), to_pil(img_b), batched=False)
print("warp", tuple(warp.shape), "certainty", tuple(certainty.shape))

# Dense correspondence: resample B into A's frame

`warp[..., 2:4]` is exactly a `grid_sample` grid: for every pixel of A it gives the
matched location in B. Sampling B at that grid yields `b_aligned`, pixel-for-pixel
aligned to A. No rectification, no disparity search.

In [ ]:
CERT_THRESH = 0.5   # keep only confident correspondences (lower if too few survive)

A = to01(img_a)
b_t = torch.from_numpy(to01(img_b)).permute(2, 0, 1).unsqueeze(0).float().to(device)  # (1,3,h,w)
grid = warp[..., 2:4].unsqueeze(0).float().to(device)                                 # (1,h_a,w_a,2)

b_aligned = F.grid_sample(b_t, grid, mode="bilinear", align_corners=False)[0]          # (3,h_a,w_a)
b_aligned = b_aligned.permute(1, 2, 0).cpu().numpy()                                   # (h_a,w_a,3)

cert = certainty.cpu().numpy()
mask = cert > CERT_THRESH
print(f"{int(mask.sum())} confident pixels ({100 * mask.mean():.1f}% of frame), cert>{CERT_THRESH}")

In [ ]:
A_show = np.where(mask[..., None], np.clip(A, 0, 1), 0)
B_show = np.where(mask[..., None], np.clip(b_aligned, 0, 1), 0)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(A_show); axes[0].set_title("A (confident pixels)"); axes[0].axis("off")
axes[1].imshow(B_show); axes[1].set_title("B resampled to A (RoMa dense flow)"); axes[1].axis("off")
axes[2].imshow(mask, cmap="gray"); axes[2].set_title(f"certainty > {CERT_THRESH} mask"); axes[2].axis("off")
plt.tight_layout(); plt.show()

# ΔE2000 over the dense correspondences

In [ ]:
pa = np.clip(A[mask], 0, 1).astype(np.float64)          # (N,3) RGB from A
pb = np.clip(b_aligned[mask], 0, 1).astype(np.float64)  # (N,3) RGB from B (dense-flow matched)

lab_a = rgb_to_lab(pa, linearize=False)
lab_b = rgb_to_lab(pb, linearize=False)
de = delta_e_cie2000(lab_a, lab_b)                       # (N,) per-pixel ΔE00

stats = delta_e_summary(de)
print("ΔE2000 over RoMa dense correspondences:")
for k, v in stats.items():
    print(f"  {k:7s} {v:.3f}")

In [ ]:
de_map = np.full(mask.shape, np.nan, dtype=np.float64)
de_map[mask] = de

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(de_map, cmap="inferno", vmin=0)
axes[0].set_title("ΔE00 map (dense)"); axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].hist(de, bins=40, color="steelblue")
axes[1].axvline(stats["mean"], color="crimson", label=f"mean = {stats['mean']:.2f}")
axes[1].axvline(stats["median"], color="orange", label=f"median = {stats['median']:.2f}")
axes[1].set_title("Per-pixel ΔE00"); axes[1].set_xlabel("ΔE2000"); axes[1].legend()
plt.tight_layout(); plt.show()